# Dataset preparation

This notebook demonstrates the preparation of the Sentinel-2 dataset
used for satellite image matching.

The goal is to prepare pairs of satellite images acquired during
different seasons while preserving their spatial correspondence.

The resulting dataset will be used for image matching with the LoFTR
feature matching algorithm.

## Dataset

The Deforestation in Ukraine dataset from Kaggle was selected
for this project.

The dataset contains Sentinel-2 satellite imagery acquired between 2016 and 2019.

Each scene contains a True Color Image (TCI), which is stored in
JPEG2000 (.jp2) format.

Since the task focuses on matching images rather than land cover
classification, only TCI images are used.

In [ ]:
!kaggle datasets download isaienkov/deforestation-in-ukraine --unzip

Dataset URL: https://www.kaggle.com/datasets/isaienkov/deforestation-in-ukraine
License(s): CC0-1.0
100% 32.7G/32.7G [06:05<00:00, 96.1MB/s]



In [1]:
import os
import glob
import random

from datetime import timedelta

from pathlib import Path

import shutil
import json

In [ ]:
folders = glob.glob("/content/S2*")

print("Number of Sentinel products:", len(folders))

for f in folders[:10]:
    print(os.path.basename(f))

Number of Sentinel products: 50
S2A_MSIL1C_20190427T083601_N0207_R064_T36UXA_20190427T100851
S2A_MSIL1C_20180919T083621_N0206_R064_T36UXA_20180919T105540
S2A_MSIL1C_20180731T083601_N0206_R064_T36UXA_20180731T110233
S2A_MSIL1C_20161026T083032_N0204_R021_T36UYA_20161026T083029
S2A_MSIL1C_20160502T083602_N0201_R064_T36UYA_20160502T084425
S2B_MSIL1C_20190412T083609_N0207_R064_T36UYA_20190412T122445
S2A_MSIL1C_20180810T083601_N0206_R064_T36UXA_20180810T124435
S2B_MSIL1C_20180904T083549_N0206_R064_T36UXA_20180904T123955
S2A_MSIL1C_20190904T083601_N0208_R064_T36UYA_20190904T110155
S2A_MSIL1C_20190517T083601_N0207_R064_T36UYA_20190517T100755


In [ ]:
DATASET_ROOT = Path("/content")

## Reading Sentinel-2 RGB images

Each Sentinel-2 scene contains multiple spectral bands.

For this task, only the TCI (True Color Image) is used because it
provides an RGB representation suitable for feature matching.

In [ ]:
tci_files = sorted(DATASET_ROOT.rglob("*TCI.jp2"))

print(f"Found {len(tci_files)} TCI images")

for f in tci_files[:5]:
    print(f)

Found 50 TCI images
/content/S2A_MSIL1C_20160212T084052_N0201_R064_T36UYA_20160212T084510/S2A_MSIL1C_20160212T084052_N0201_R064_T36UYA_20160212T084510.SAFE/GRANULE/L1C_T36UYA_A003350_20160212T084510/IMG_DATA/T36UYA_20160212T084052_TCI.jp2
/content/S2A_MSIL1C_20160330T082542_N0201_R021_T36UYA_20160330T082810/S2A_MSIL1C_20160330T082542_N0201_R021_T36UYA_20160330T082810.SAFE/GRANULE/L1C_T36UYA_A004022_20160330T082810/IMG_DATA/T36UYA_20160330T082542_TCI.jp2
/content/S2A_MSIL1C_20160405T085012_N0201_R107_T36UYA_20160405T085012/S2A_MSIL1C_20160405T085012_N0201_R107_T36UYA_20160405T085012.SAFE/GRANULE/L1C_T36UYA_A004108_20160405T085012/IMG_DATA/T36UYA_20160405T085012_TCI.jp2
/content/S2A_MSIL1C_20160502T083602_N0201_R064_T36UYA_20160502T084425/S2A_MSIL1C_20160502T083602_N0201_R064_T36UYA_20160502T084425.SAFE/GRANULE/L1C_T36UYA_A004494_20160502T084425/IMG_DATA/T36UYA_20160502T083602_TCI.jp2
/content/S2A_MSIL1C_20160509T082612_N0202_R021_T36UYA_20160509T083548/S2A_MSIL1C_20160509T082612_N0202_R

## Organizing image metadata

The acquisition date and Sentinel-2 tile identifier are extracted from each image filename.

This information is stored in a table to simplify chronological sorting and seasonal pair selection.

In [ ]:
import pandas as pd

records = []

for file in tci_files:

    name = file.stem
    # T36UYA_20160618T082602_TCI

    parts = name.split("_")

    tile = parts[0]
    date = parts[1][:8]

    records.append({
        "tile": tile,
        "date": pd.to_datetime(date),
        "path": file
    })

df = pd.DataFrame(records)

df.head()

,tile,date,path
0,T36UYA,2016-02-12,/content/S2A_MSIL1C_20160212T084052_N0201_R064...
1,T36UYA,2016-03-30,/content/S2A_MSIL1C_20160330T082542_N0201_R021...
2,T36UYA,2016-04-05,/content/S2A_MSIL1C_20160405T085012_N0201_R107...
3,T36UYA,2016-05-02,/content/S2A_MSIL1C_20160502T083602_N0201_R064...
4,T36UYA,2016-05-09,/content/S2A_MSIL1C_20160509T082612_N0202_R021...


In [ ]:
df = df.sort_values(["tile", "date"])

## Creating seasonal image pairs

Images are grouped by Sentinel-2 tile to ensure that only the same
geographical region is compared.

Pairs are selected with an acquisition time difference of approximately 180 days (±30 days), increasing seasonal variability while preserving spatial correspondence.

In [ ]:
TARGET_DAYS = 180
TOLERANCE = 30

In [ ]:
pairs = []

for tile, group in df.groupby("tile"):

    group = group.sort_values("date").reset_index(drop=True)

    for i in range(len(group)):

        date1 = group.loc[i, "date"]

        for j in range(i + 1, len(group)):

            date2 = group.loc[j, "date"]

            diff = abs((date2 - date1).days)

            if abs(diff - TARGET_DAYS) <= TOLERANCE:

                pairs.append({

                    "tile": tile,

                    "date1": date1,

                    "date2": date2,

                    "days": diff,

                    "image1": group.loc[i, "path"],

                    "image2": group.loc[j, "path"]

                })

In [ ]:
pairs_df = pd.DataFrame(pairs)

pairs_df.head()

,tile,date1,date2,days,image1,image2
0,T36UYA,2016-02-12,2016-08-30,200,/content/S2A_MSIL1C_20160212T084052_N0201_R064...,/content/S2A_MSIL1C_20160830T083602_N0204_R064...
1,T36UYA,2016-03-30,2016-08-30,153,/content/S2A_MSIL1C_20160330T082542_N0201_R021...,/content/S2A_MSIL1C_20160830T083602_N0204_R064...
2,T36UYA,2016-03-30,2016-10-26,210,/content/S2A_MSIL1C_20160330T082542_N0201_R021...,/content/S2A_MSIL1C_20161026T083032_N0204_R021...
3,T36UYA,2016-04-05,2016-10-26,204,/content/S2A_MSIL1C_20160405T085012_N0201_R107...,/content/S2A_MSIL1C_20161026T083032_N0204_R021...
4,T36UYA,2016-05-02,2016-10-26,177,/content/S2A_MSIL1C_20160502T083602_N0201_R064...,/content/S2A_MSIL1C_20161026T083032_N0204_R021...


## Preparing the final dataset

The first 30 seasonal pairs are selected for the image matching task.

Each pair is stored in a separate directory together with a metadata file containing acquisition dates and tile information.

In [ ]:
OUTPUT_DIR = Path("/prepared_dataset")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
pairs_df = pairs_df.head(30).copy()

print(f"Selected {len(pairs_df)} seasonal pairs")

Selected 30 seasonal pairs


In [ ]:
for idx, row in pairs_df.iterrows():

    pair_name = f"pair_{row['date1'].strftime('%Y%m%d')}_{row['date2'].strftime('%Y%m%d')}"
    pair_dir = OUTPUT_DIR / pair_name
    pair_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy2(row["image1"], pair_dir / "image_A.jp2")
    shutil.copy2(row["image2"], pair_dir / "image_B.jp2")

    metadata = {
        "tile": row["tile"],
        "date_A": str(row["date1"].date()),
        "date_B": str(row["date2"].date()),
        "days_between": int(row["days"])
    }

    with open(pair_dir / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=4)

print("Dataset created!")

Dataset created!


## Dataset structure

The prepared dataset has the following structure:

```
prepared_dataset/
│
├── pair_YYYYMMDD_YYYYMMDD/
│   ├── image_A.jp2
│   ├── image_B.jp2
│   └── metadata.json
│
└── ...
```

This directory structure is used during the inference stage of the
matching algorithm.